# Comparing Three Versions of `load_clean_resp_signal`

In this notebook, we developed and refined three versions of a function to load, filter, and downsample respiration signals from `.h5` files.  
Each version differs in how it handles **sampling rate detection**, **downsampling**, and **band-pass filtering**.

---

## **Version 1: Dynamic fs + NeuroKit band-pass**
- **Sampling rate (`fs`)**: Calculated from metadata (`len(resp) / duration_sec`).  
- **Anti-alias filter**: Low-pass cutoff at `(target_rate/2) / (fs/2)`.  
- **Downsampling**: Polyphase (`resample_poly`) with integer ratio.  
- **Band-pass filtering**: `0.1–20 Hz` using **NeuroKit2’s `signal_filter`**.  
- ✅ **Pros**: Flexible to any acquisition rate, easy filtering with NeuroKit.  
- ⚠️ **Cons**: Relies on NeuroKit, and integer-only downsampling can slightly misalign signal length.

---

## **Version 2: Fixed fs (20 kHz) + SciPy band-pass**
- **Sampling rate (`fs`)**: Hard-coded to `20,000 Hz`.  
- **Anti-alias filter**: Explicit low-pass before decimation.  
- **Downsampling**: Polyphase (`resample_poly`) based on integer factor.  
- **Band-pass filtering**: `0.1–20 Hz` via manual SciPy Butterworth.  
- ✅ **Pros**: Self-contained (no NeuroKit dependency), full control over filters.  
- ⚠️ **Cons**: Assumes fixed sampling rate (unsafe if files differ), still integer-ratio resampling.

---

## **Version 3: Adaptive anti-alias + exact resampling**
- **Sampling rate (`fs`)**: Dynamically computed (`len(resp) / duration_sec`).  
- **Anti-alias filter**: Smarter logic:
  - If `target_rate >= fs`: low-pass at ~20 Hz (denoising only).  
  - If `target_rate < fs`: cutoff at `target_rate/2`.  
  - Safely clamps normalized cutoff (`Wn`) between `0.0001–0.9999`.  
- **Downsampling**: `scipy.signal.resample` with **exact output length** (`duration_sec * target_rate`).  
  - Prevents floor/rounding errors from integer-only ratios.  
- **Band-pass filtering**: `0.1–15 Hz` using NeuroKit (more conservative).  
- ✅ **Pros**: Safest and most general — adaptive to any fs, exact alignment, cleaner high-frequency rejection.  
- ⚠️ **Cons**: Still depends on NeuroKit for the final filtering stage.

---

## **Summary**
- **Use Version 1** if you want quick and flexible filtering with NeuroKit.  
- **Use Version 2** if you always record at 20 kHz and want a fully SciPy-based pipeline.  
- **Use Version 3** if you need the most **robust and general solution** — works across sampling rates, guarantees correct output length, and applies a tighter physiological band.

Next steps:  
👉 We could merge the best of Version 3 with the SciPy-only band-pass from Version 2 to create a **dependency-free, adaptive, and precise pipeline**.


In [1]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, resample_poly
import neurokit2 as nk
from scipy.signal import find_peaks
import seaborn as sns
from scipy.stats import wilcoxon

# Pandas display settings (optional)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

In [2]:
# RESP FILE PATHS
RI1_3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5"
RI2_3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5"
BLRI_s3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5"
BLRI_s4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5"
RI1_4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5"
RI2_4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5"
RI1_2_3_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5"
RI2_2_3_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5"
RI1_4_8_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5"
RI2_4_8_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5"
RI1_1_1_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5"
RI2_1_1_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5"
RI1_1_2_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5"
RI2_1_2_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5"
RI1_2_4_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5"
RI2_2_4_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5"
RI2_3_5_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5"
RI1_3_5_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5"

In [3]:
ri1_h5_paths = {
    "RI1_3_6": RI1_3_6_h5_path,
    "RI1_4_7": RI1_4_7_h5_path,  
    "RI1_2_3": RI1_2_3_h5_path,
    "RI1_4_8": RI1_4_8_h5_path,
    "RI1_1_1": RI1_1_1_h5_path,
    "RI1_1_2": RI1_1_2_h5_path,
    "RI1_2_4": RI1_2_4_h5_path,
    "RI1_3_5": RI1_3_5_h5_path,
}

In [4]:
ri2_h5_paths = {
    "RI2_3_6": RI2_3_6_h5_path,
    "RI2_4_7": RI2_4_7_h5_path,
    "RI2_2_3": RI2_2_3_h5_path,
    "RI2_4_8": RI2_4_8_h5_path,
    "RI2_1_1": RI2_1_1_h5_path,
    "RI2_1_2": RI2_1_2_h5_path,
    "RI2_2_4": RI2_2_4_h5_path,
    "RI2_3_5": RI2_3_5_h5_path,
}



In [5]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Loads, filters, downsamples respiration from .h5, returns cleaned signal and time vector.
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()
            ekg_meta = dict(f['ekg_metadata'].attrs)
        duration_sec = ekg_meta['duration_sec']
        fs = len(resp) / duration_sec
    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None

    # Pre-filter before downsampling
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # Downsample
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # Bandpass filter with neurokit
    rsp_cleaned = nk.signal_filter(
        downsampled,
        lowcut=0.1,
        highcut=20,
        method="butterworth",
        sampling_rate=target_rate,
        order=2
    )

    # Generate matching time vector
    time_vector = np.arange(len(rsp_cleaned)) / target_rate

    return rsp_cleaned, time_vector, target_rate

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
from scipy.signal import butter, filtfilt, resample_poly

def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Loads, filters, downsamples respiration from .h5, 
    returns cleaned signal and time vector.
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()
            ekg_meta = dict(f['ekg_metadata'].attrs)
        duration_sec = ekg_meta['duration_sec']
        fs = 20000 # Assuming a fixed sampling rate of 20 kHz
    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None

    # Pre-filter before downsampling (low-pass to avoid aliasing)
    nyquist = fs / 2
    norm_cutoff = (target_rate / 2) / nyquist
    b, a = butter(N=4, Wn=norm_cutoff, btype='low')
    filtered_resp = filtfilt(b, a, resp)

    # Downsample
    downsample_factor = int(fs // target_rate)
    downsampled = resample_poly(filtered_resp, up=1, down=downsample_factor)

    # Bandpass filter (0.1–20 Hz)
    nyquist_new = target_rate / 2
    b, a = butter(2, [0.1/nyquist_new, 20/nyquist_new], btype='band')
    rsp_cleaned = filtfilt(b, a, downsampled)

    # Generate matching time vector
    time_vector = np.arange(len(rsp_cleaned)) / target_rate
    return rsp_cleaned, time_vector, target_rate


def plot_first_20s(h5_paths, label_prefix):
    plt.figure(figsize=(15, 10))
    for i, (key, path) in enumerate(h5_paths.items(), 1):
        rsp_cleaned, time_vector, fs = load_clean_resp_signal(path)
        if rsp_cleaned is None:
            continue

        # Restrict to first 20 seconds
        mask = time_vector <= 5
        plt.subplot(len(h5_paths)//2, 2, i)
        plt.plot(time_vector[mask], rsp_cleaned[mask])
        plt.title(f"{label_prefix} {key}")
        plt.xlabel("Time (s)")
        plt.ylabel("Respiration (a.u.)")

    plt.tight_layout()
    plt.show()


In [7]:
def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Load respiration from .h5, anti-alias LPF at raw fs, resample to target_rate,
    band-pass clean, and return (signal, time, fs_out).
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()
            ekg_meta = dict(f['ekg_metadata'].attrs)
        duration_sec = float(ekg_meta['duration_sec'])
        fs = len(resp) / duration_sec
    except Exception as e:
        print(f"Error loading {h5_file}: {e}")
        return None, None, None

    # --- Anti-alias LPF at raw fs ---
    if target_rate >= fs:
        # No decimation needed; still low-pass a bit to remove HF noise
        cutoff_hz = min(0.45 * fs, 20.0)  # keep it sensible if fs is small
        Wn = cutoff_hz / (fs / 2.0)
    else:
        cutoff_hz = target_rate / 2.0       # classic anti-alias cutoff
        Wn = cutoff_hz / (fs / 2.0)

    Wn = min(max(Wn, 0.0001), 0.9999)       # clamp for safety
    b, a = butter(N=4, Wn=Wn, btype='low')
    filtered = filtfilt(b, a, resp)

    # --- Resample to EXACT target_rate ---
    # Prefer exact-length resample to avoid floor issues
    N_out = int(round(duration_sec * target_rate))
    # If you prefer polyphase, use Fraction to get integers:
    # from fractions import Fraction
    # frac = Fraction(target_rate, int(round(fs))).limit_denominator()
    # resampled = resample_poly(filtered, up=frac.numerator, down=frac.denominator)
    resampled = scipy.signal.resample(filtered, N_out)

    # --- Final band-pass clean at the new fs ---
    # Keep physiological band; adjust if you know your spectrum well
    lowcut, highcut = 0.1, 15.0
    rsp_cleaned = nk.signal_filter(
        resampled, lowcut=lowcut, highcut=highcut,
        method="butterworth", sampling_rate=target_rate, order=2
    )

    time_vector = np.arange(len(rsp_cleaned)) / float(target_rate)
    return rsp_cleaned, time_vector, float(target_rate)


In [8]:
(ri1_h5_paths, "RI1")
(ri2_h5_paths, "RI2")


({'RI2_3_6': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5',
  'RI2_4_7': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5',
  'RI2_2_3': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5',
  'RI2_4_8': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5',
  'RI2_1_1': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5',
  'RI2_1_2': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5',
  'RI2_2_4': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5',
  'RI2_3_5': 'E:\\Aim1\\AIM1\\Day1_new\\resp_h5\\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5'},
 'RI2')